In [40]:
# Check that the libraries we need are installed

import pandas    # pandas: reads and handles tables of data 
import sklearn   # scikit-learn: machine learning models, train/test split, error metrics
import joblib    # joblib: saves a trained model to a file so the app can reuse it

print("ok", sklearn.__version__)


ok 1.9.0


In [41]:
# Load the listings
# Synthetic test data generated with AI (Claude), see AI log
import pandas as pd

df = pd.read_csv("data/test_listings.csv")

print(df.shape)   # (number of rows, number of columns)
df.head()         # first 5 rows

(300, 8)


,id,municipality,rooms,surface_m2,year_built,lat,lon,rent_gross
0,1,Dietikon,1.5,42,1963,47.40594,8.41217,1130
1,2,Zürich,3.5,93,2014,47.37935,8.54354,3650
2,3,Thalwil,4.0,90,1985,47.30273,8.54879,2860
3,4,Horgen,4.0,96,1993,47.25015,8.59390,2900
4,5,Winterthur,3.0,76,2008,47.49967,8.73789,2080


In [42]:
# Basic statistics for each numeric column (count, mean, min, max, ...)
df.describe()

,id,rooms,surface_m2,year_built,lat,lon,rent_gross
count,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000
mean,150.500000,3.368333,83.443333,1988.906667,47.374146,8.573030,2443.066667
std,86.746758,0.966061,22.089362,22.413072,0.080988,0.093283,746.692996
min,1.000000,1.500000,26.000000,1950.000000,47.215490,8.386010,760.000000
25%,75.750000,2.500000,68.000000,1968.000000,47.302948,8.528688,1920.000000
50%,150.500000,3.500000,85.000000,1990.000000,47.377745,8.553110,2340.000000
75%,225.250000,4.000000,99.250000,2009.000000,47.408613,8.633448,2870.000000
max,300.000000,5.500000,144.000000,2025.000000,47.530410,8.738270,5950.000000


In [43]:
# Average rent per municipality
groups = df.groupby("municipality")       # 1. split the table into one group per municipality
rents = groups["rent_gross"]              # 2. keep only the rent column
average_rent = rents.mean()               # 3. average rent of each group
average_rent.sort_values(ascending=False) # 4. most expensive first

municipality
Zürich                2903.972603
Adliswil              2718.421053
Thalwil               2688.947368
Dübendorf             2624.615385
Horgen                2515.000000
Meilen                2423.636364
Uster                 2293.888889
Bülach                2248.750000
Wallisellen           2235.000000
Winterthur            2220.588235
Schlieren             2196.363636
Kloten                2099.166667
Wädenswil             2037.692308
Dietikon              1997.619048
Affoltern am Albis    1830.625000
Name: rent_gross, dtype: float64

In [44]:
#  Average rent per m² per municipality 

df["m2_price"] = df["rent_gross"] / df["surface_m2"]       # 1. new column: CHF per m² for each flat
groups = df.groupby("municipality")                        # 2. regroup so the new column is included                               
avg_rent_m2 = groups["m2_price"].mean()                    # 3. average CHF/m² per municipality
avg_rent_m2.sort_values(ascending=False).round(2)          # 4. most expensive first (rounded)

municipality
Zürich                36.22
Meilen                31.22
Dübendorf             31.06
Thalwil               30.93
Adliswil              30.26
Wallisellen           28.97
Horgen                28.94
Schlieren             27.60
Uster                 26.57
Kloten                26.52
Dietikon              26.30
Wädenswil             26.21
Bülach                25.97
Winterthur            24.57
Affoltern am Albis    23.97
Name: m2_price, dtype: float64

In [45]:
X = df.drop(["id", "municipality", "rent_gross", "m2_price"], axis=1)
y = df["m2_price"]
print(X.columns)

Index(['rooms', 'surface_m2', 'year_built', 'lat', 'lon'], dtype='str')


In [46]:
# Random Forest: predicts the price per m2, then rent = price per m2 x m2
# Import model functions
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Create the model (random_state makes the results reproducible)
model = RandomForestRegressor(random_state=1)

# Split the data: 70% training, 30% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
print(X_train.shape, X_test.shape)

# Train the model on the training data only
model.fit(X_train, y_train)

# Predict the price per m2 of the test apartments (the model has never seen them)
y_pred = model.predict(X_test)

# Clearer names for the test results
real_price_m2 = y_test                    # actual price per m2 (from the data)
predicted_price_m2 = y_pred               # price per m2 estimated by the model
surface = X_test["surface_m2"]            # size of each test apartment

# Convert price per m2 into monthly rent in CHF
real_rent = real_price_m2 * surface
predicted_rent = predicted_price_m2 * surface

# How far are the estimated rents from the real ones, in CHF?
mae = mean_absolute_error(real_rent, pred_rent)        # average error
rmse = root_mean_squared_error(real_rent, pred_rent)   # big errors count more

print("MAE: ", round(mae), "CHF")
print("RMSE:", round(rmse), "CHF")

(210, 5) (90, 5)
MAE:  193 CHF
RMSE: 241 CHF


In [47]:
# Baseline: a simple method without machine learning.
# Estimated rent = average price per m2 of the municipality x m2 of the apartment.
# If our Random Forest is not better than this, the model is not worth using.

# 1. Split the full table with the same parameters as the model.
#    X has no municipality and no rent, but the baseline needs both, so we split df.
#    Same test_size and random_state -> same 90 test apartments as the model,
#    so the two errors can be compared fairly.
train_df, test_df = train_test_split(df, test_size=0.3, random_state=1)

# 2. Average price per m2 of each municipality.
#    Training apartments only: the test rents must stay unseen, like for the model.
avg_price_m2 = train_df.groupby("municipality")["m2_price"].mean()

# 3. Estimated rent of each test apartment.
#    .map() replaces each municipality with its average price per m2 from step 2
#    ("Uster" -> 26.6), then we multiply by the apartment's m2.
#    Example: 80 m2 in Uster -> 26.6 x 80 = 2'128 CHF
baseline_rent = test_df["municipality"].map(avg_price_m2) * test_df["surface_m2"]

# 4. Mean Absolute Error (MAE): for each test apartment, the difference between
#    real and estimated rent (without sign), then the average.
#    Same metric as the model, so the two numbers can be compared directly.
mae_baseline = mean_absolute_error(test_df["rent_gross"], baseline_rent)

print("Baseline MAE:", round(mae_baseline), "CHF")
print("Model MAE:   ", round(mae), "CHF")

Baseline MAE: 258 CHF
Model MAE:    193 CHF


### Result: model vs. baseline

| Method | MAE (average error) |
|---|---|
| Baseline: average price per m² of the municipality × m² | 258 CHF |
| Random Forest: predicted price per m² × m² | **193 CHF** |

On the same 90 test apartments, the Random Forest misses the real rent by **193 CHF on average**, against 258 CHF for the simple calculation. That is about **25% more precise**.

Why the model is better: the baseline only uses the municipality and the size. The Random Forest also uses the number of rooms, the year of construction and the exact location (latitude and longitude), so it can tell apart, for example, a new apartment near the lake from an old one at the edge of the same municipality.

We will use the model's MAE as the tolerance for the verdict: if the asking rent differs from the fair rent by less than this amount, the apartment is rated as "fair".

*Note: these results come from the synthetic test dataset (generated with AI, see AI log). They will be recalculated with the real Flatfox data.*